# Elements and Lattices

PyTao provides Pydantic models that let you query Tao element data into structured Python objects.
The two main classes are `Element` (a single lattice element) and `Lattice` (a collection of elements).

## Setup

In [ ]:
from pytao import Tao

tao = Tao(init_file="$ACC_ROOT_DIR/bmad-doc/tao_examples/cesr/tao.init", plot="mpl")

## Getting a single Element

Use `tao.ele()` to get one element. By default, it loads a comprehensive set of data (twiss, orbit, attributes, floor coordinates, etc.).

In [ ]:
ele = tao.ele("Q00W")
ele.name, ele.key

### Head information

Every element has a `head` with metadata: name, key (element type), position, indices, and capability flags.

In [ ]:
ele.head

### General attributes

General attributes are element-type-specific parameters like `L` (length), `k1` (quadrupole strength), etc.

In [ ]:
ele.attrs

In [ ]:
# Access individual attributes (case-insensitive)
ele.attrs["L"]

### Twiss parameters

In [ ]:
ele.twiss

### Orbit

In [ ]:
ele.orbit

### Floor coordinates

In [ ]:
ele.floor

### Transfer matrix (mat6)

In [ ]:
ele.mat6

## Getting multiple Elements

Use `tao.eles()` to query multiple elements at once. It supports names, indices, ranges, wildcards, and key-based matching.

In [ ]:
# By index range
elements = tao.eles("1:5")
for ele in elements:
    print(f"{ele.name:20s} key={ele.key:15s} s={ele.head.s:.3f}")

In [ ]:
# By wildcard
quads = tao.eles("Q0*")
for q in quads:
    print(f"{q.name:20s} K1={q.attrs['K1'].data}")

In [ ]:
# By element key - get all quadrupoles
all_quads = tao.eles("quad::*")
print(f"Found {len(all_quads)} quadrupoles")

## Controlling what data is loaded

Loading all data for many elements can be slow. You can control which fields are loaded.

In [ ]:
# Minimal element - only head data
ele_minimal = tao.ele("Q00W", defaults=False)
print(f"twiss: {ele_minimal.twiss}")
print(f"orbit: {ele_minimal.orbit}")

In [ ]:
# Only specific fields
ele_twiss_only = tao.ele("Q00W", defaults=False, twiss=True, attrs=True)
print(f"twiss: {ele_twiss_only.twiss is not None}")
print(f"orbit: {ele_twiss_only.orbit is not None}")

In [ ]:
# Defaults on, but exclude orbit
ele_no_orbit = tao.ele("Q00W", orbit=False)
print(f"twiss: {ele_no_orbit.twiss is not None}")
print(f"orbit: {ele_no_orbit.orbit is not None}")

You can also change the global defaults for all future queries:

In [ ]:
from pytao.model import Element

# See current defaults
print("Current defaults:", Element.DEFAULTS)

## Lattice objects

A `Lattice` is a collection of `Element`s. There are several constructors depending on what subset of the lattice you need.

In [ ]:
from pytao.model import Lattice

# Load tracking elements (skips lords/markers)
lat = Lattice.from_tao_tracking(tao)
print(lat)

In [ ]:
# Load unique (non-slave) elements
lat_unique = Lattice.from_tao_unique(tao)
print(lat_unique)

In [ ]:
# Load a subset by range
lat_subset = Lattice.from_tao_tracking(tao, track_start="1", track_end="20")
print(lat_subset)

### Accessing elements in a Lattice

Lattice objects provide convenient dictionaries for lookup.

In [ ]:
# By name
by_name = lat.by_element_name
list(by_name.keys())[:10]

In [ ]:
# By element type (key)
by_key = lat.by_element_key
print("Element types:", list(by_key.keys()))

In [ ]:
# By index
by_index = lat.by_element_index
by_index[1].name

## Plotting with Element data

Since all data is in Python objects, it's easy to plot with matplotlib or any other library.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Extract Twiss data from lattice elements
eles = lat.elements
s_values = [ele.head.s for ele in eles]
alpha_a = [ele.twiss.alpha_a for ele in eles]
alpha_b = [ele.twiss.alpha_b for ele in eles]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(s_values, alpha_a, label=r"$\alpha_a$")
ax.plot(s_values, alpha_b, label=r"$\alpha_b$")
ax.set_xlabel("s [m]")
ax.set_ylabel(r"$\alpha_A$, $\alpha_B$ [m]")
ax.set_title("Twiss Alpha Functions")
ax.legend()
plt.tight_layout()

In [ ]:
tao.plot("alpha", width=12)

## Saving and loading

Element and Lattice objects can be serialized to JSON, compressed JSON, or msgpack (the fastest!).

In [ ]:
from pathlib import Path

lat_subset = Lattice.from_tao_tracking(tao, track_start="1", track_end="20")

# Save to JSON
lat_subset.write(Path("lattice_subset.json"))

# Save to compressed JSON
lat_subset.write(Path("lattice_subset.json.gz"))

# Save to msgpack
lat_subset.write(Path("lattice_subset.msgpack"))

In [ ]:
# Load back from any format

for path in ["lattice_subset.json", "lattice_subset.json.gz", "lattice_subset.msgpack"]:
    print("Loading:", path)
    lat_loaded = Lattice.from_file(path)
    print(lat_loaded)
    print("Equal to original?", lat_loaded == lat_subset)
    print()

In [ ]:
# Cleanup
for fn in ["lattice_subset.json", "lattice_subset.json.gz", "lattice_subset.msgpack"]:
    fn = Path(fn)
    if fn.exists():
        fn.unlink()